In [1]:
import os
from functools import partial
from typing import Literal
from pydantic import BaseModel

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch

# 1. Initialize with an active, valid API model identifier
llm_kimi = ChatOpenAI(
    api_key=os.environ.get('KIMI_API_KEY'),
    base_url='https://moonshot.ai',
    model='moonshot-v1-8k'
)

# 2. Schema and Output Parser Definition
class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = PydanticOutputParser(pydantic_object=llm_schema)

# 3. Request Generation Prompt with Parser Schema Built In
initial_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a great movie reviewer.\n{format_instructions}'),
    ('user', 'Based on the input, categorize movies as positive or negative: {input}')
]).partial(format_instructions=llm_structured_output.get_format_instructions())

# 4. Helper Lambdas and Subchains
def pydantic_json(parsed_output: llm_schema):
    return parsed_output.movie_summary_flag

pydantic_json_lambda = RunnableLambda(pydantic_json)
str_parser = StrOutputParser()

def social_media_post_generator(text: str, platform):
    post_template = ChatPromptTemplate.from_messages([
        ('system', 'You are the best social media handler. You write catchy posts for {platform}'),
        ('user', 'Write the best post possible for the {text} for the platform {platform}')
    ])
    chain = post_template | llm_kimi | str_parser
    return chain.invoke({'text': text, 'platform': platform})

linkedin_chain = RunnableLambda(partial(social_media_post_generator, platform='Linkedin'))
insta_chain = RunnableLambda(partial(social_media_post_generator, platform='Instagram'))

# 5. Routing Condition Branch
conditional_chain = RunnableBranch(
    (lambda x: 'positive' in x, linkedin_chain),
    insta_chain
)

# 6. Combined Execution Chain Link
final_orchestrator = (
    initial_template | 
    llm_kimi | 
    llm_structured_output | 
    pydantic_json_lambda | 
    conditional_chain
)

# 7. Execute Chain
result = final_orchestrator.invoke({"input": "KGF was a great movie"})
print(result)


AttributeError: 'str' object has no attribute 'model_dump'